## Example 1: FrameInfo from Traceback

In [ ]:
# Example 1: FrameInfo from Traceback
import stack_data
import sys

print("=== FrameInfo from Traceback ===")

def inner_func(x):
    y = x * 2
    raise ValueError(f"Test error with x={x}")

try:
    inner_func(5)
except ValueError:  # raised on purpose above; any OTHER exception must fail this example
    tb = sys.exc_info()[2]
    while tb.tb_next:  # walk to the frame that raised
        tb = tb.tb_next
    # FrameInfo takes the traceback itself -- its second parameter is `options`, not a line number.
    frame_info = stack_data.FrameInfo(tb)
    
    print(f"Function: {frame_info.code.co_name}")
    print(f"Line: {frame_info.lineno}")
    print(f"Filename: {frame_info.code.co_filename.split('/')[-1]}")
    print(f"\nUse case: Extract stack info from exceptions")
    current = [line.text for line in frame_info.lines if getattr(line, "is_current", False)]
    assert frame_info.code.co_name == "inner_func", frame_info.code.co_name
    assert current and "raise ValueError" in current[0], current
else:
    raise AssertionError("inner_func(5) did not raise -- there is nothing for stack_data to inspect")

print("\nExample 1 completed!")

## Example 2: Variable Extraction

In [ ]:
# Example 2: Variable Extraction
import stack_data
import sys

print("=== Variable Extraction ===")

def calculate(a, b, c):
    result = a * b + c
    intermediate = result ** 2
    raise RuntimeError("Demo error")

try:
    calculate(3, 4, 5)
except RuntimeError:  # raised on purpose above; any OTHER exception must fail this example
    tb = sys.exc_info()[2]
    while tb.tb_next:  # the variables live in the frame that raised
        tb = tb.tb_next
    frame_info = stack_data.FrameInfo(tb)
    
    print("Variables in frame:")
    for var in frame_info.variables:
        print(f"  {var.name} = {var.value}")
    
    print(f"\nUse case: Inspect variables at error point")
    values = {var.name: var.value for var in frame_info.variables}
    expected = {"a": 3, "b": 4, "c": 5, "result": 17, "intermediate": 289}
    assert expected.items() <= values.items(), values
else:
    raise AssertionError("calculate(3, 4, 5) did not raise -- there is nothing for stack_data to inspect")

print("\nExample 2 completed!")

## Example 3: Source Code Lines

In [ ]:
# Example 3: Source Code Lines
import stack_data
import sys

print("=== Source Code Lines ===")

def process_data(items):
    total = sum(items)
    average = total / len(items)
    return average / 0  # Division by zero

try:
    process_data([1, 2, 3])
except ZeroDivisionError:  # raised on purpose above; any OTHER exception must fail this example
    tb = sys.exc_info()[2]
    while tb.tb_next:  # walk to the frame that raised
        tb = tb.tb_next
    frame_info = stack_data.FrameInfo(tb)
    
    print(f"Function: {frame_info.code.co_name}")
    print(f"Source lines available: {len(frame_info.source.lines)}")
    print(f"Error at line {frame_info.lineno}")
    
    print(f"\nUse case: Access source code context")
    assert frame_info.code.co_name == "process_data", frame_info.code.co_name
    assert "return average / 0" in frame_info.source.lines[frame_info.lineno - 1], frame_info.lineno
else:
    raise AssertionError("process_data([1, 2, 3]) did not raise -- there is nothing for stack_data to inspect")

print("\nExample 3 completed!")

## Example 4: Options Configuration

In [ ]:
# Example 4: Options Configuration
import stack_data

print("=== Options Configuration ===")

# Create options for frame info
options = stack_data.Options(
    before=2,  # Lines before error
    after=2    # Lines after error
)

print(f"Options configured:")
print(f"  Lines before: {options.before}")
print(f"  Lines after: {options.after}")
print(f"  Type: {type(options).__name__}")

print(f"\nUse case: Configure stack frame display")

print("\nExample 4 completed!")

## Example 5: Multiple Stack Frames

In [ ]:
# Example 5: Multiple Stack Frames
import stack_data
import sys

print("=== Multiple Stack Frames ===")

def level1():
    return level2()

def level2():
    return level3()

def level3():
    raise RuntimeError("Error in level3")

try:
    level1()
except RuntimeError:  # raised on purpose above; any OTHER exception must fail this example
    tb = sys.exc_info()[2]
    frames = []
    while tb:
        frames.append(stack_data.FrameInfo(tb))  # the traceback itself, not (frame, lineno)
        tb = tb.tb_next
    
    print(f"Stack frames: {len(frames)}")
    for i, frame in enumerate(frames):
        print(f"  {i}: {frame.code.co_name} (line {frame.lineno})")
    
    print(f"\nUse case: Walk full exception stack")
    names = [frame.code.co_name for frame in frames]
    assert names == ["<module>", "level1", "level2", "level3"], names
else:
    raise AssertionError("level1() did not raise -- there is nothing for stack_data to inspect")

print("\nExample 5 completed!")